# Analyse des ventes

Ce notebook explore les ventes préparées, calcule les principaux indicateurs commerciaux et trace leur évolution. Il reprend les analyses du script `scripts/analyse_ventes.py`.

Avant de commencer, exécutez `scripts/nettoyage_ventes.py` depuis la racine du projet si `data/ventes_propres.csv` n'existe pas encore. Exécutez ensuite les cellules dans l'ordre.

## 1. Importer les bibliothèques et charger les données

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Le notebook peut être lancé depuis la racine du projet ou depuis notebooks/.
racine_projet = Path.cwd()
if not (racine_projet / "data").exists():
    racine_projet = racine_projet.parent

chemin_donnees = racine_projet / "data" / "ventes_propres.csv"
dossier_images = racine_projet / "images"
dossier_images.mkdir(parents=True, exist_ok=True)

if not chemin_donnees.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {chemin_donnees}. "
        "Lancez d'abord scripts/nettoyage_ventes.py depuis la racine du projet."
    )

df = pd.read_csv(chemin_donnees, parse_dates=["date_commande"])
df.head()

## 2. Explorer les données

Vérifions la taille du jeu de données, les types de colonnes et la présence éventuelle de valeurs manquantes.

In [ ]:
print(f"Nombre de lignes : {len(df)}")
print(f"Nombre de colonnes : {len(df.columns)}")
display(df.dtypes.to_frame(name="Type"))
display(df.isna().sum().to_frame(name="Valeurs manquantes"))

## 3. Indicateurs clés

Le chiffre d'affaires correspond à la somme de `montant_total`. Le panier moyen est calculé comme le montant moyen par ligne de commande.

In [ ]:
chiffre_affaires_total = df["montant_total"].sum()
nombre_commandes = df["commande_id"].count()
panier_moyen = df["montant_total"].mean()

print(f"Chiffre d'affaires total : {chiffre_affaires_total:,.0f} FCFA")
print(f"Nombre total de commandes : {nombre_commandes}")
print(f"Panier moyen : {panier_moyen:,.0f} FCFA")

## 4. Chiffre d'affaires par produit et par ville

In [ ]:
ventes_par_produit = (
    df.groupby("produit")["montant_total"]
    .sum()
    .sort_values(ascending=False)
)
ventes_par_ville = (
    df.groupby("ville")["montant_total"]
    .sum()
    .sort_values(ascending=False)
)

display(ventes_par_produit.to_frame(name="Chiffre d'affaires (FCFA)"))
display(ventes_par_ville.to_frame(name="Chiffre d'affaires (FCFA)"))

print("Ville générant le plus de revenus :")
display(ventes_par_ville.head(1).to_frame(name="Chiffre d'affaires (FCFA)"))

## 5. Meilleurs clients

In [ ]:
ventes_par_client = (
    df.groupby("client")["montant_total"]
    .sum()
    .sort_values(ascending=False)
)

display(ventes_par_client.to_frame(name="Chiffre d'affaires (FCFA)"))
print("Top 3 clients :")
display(ventes_par_client.head(3).to_frame(name="Chiffre d'affaires (FCFA)"))

## 6. Visualiser et enregistrer les résultats

Les trois graphiques sont affichés dans le notebook et enregistrés dans le dossier `images/`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ventes_par_produit.plot(kind="bar", ax=ax, title="Chiffre d'affaires par produit")
ax.set_ylabel("Montant en FCFA")
ax.set_xlabel("Produit")
fig.tight_layout()
fig.savefig(dossier_images / "ventes_par_produit.png")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ventes_par_ville.plot(kind="bar", ax=ax, title="Chiffre d'affaires par ville")
ax.set_ylabel("Montant en FCFA")
ax.set_xlabel("Ville")
fig.tight_layout()
fig.savefig(dossier_images / "ventes_par_ville.png")
plt.show()
plt.close(fig)

evolution = (
    df.groupby(df["date_commande"].dt.date)["montant_total"]
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
evolution.plot(kind="line", marker="o", ax=ax, title="Évolution du chiffre d'affaires")
ax.set_ylabel("Montant en FCFA")
ax.set_xlabel("Date de commande")
fig.tight_layout()
fig.savefig(dossier_images / "evolution_chiffre_affaires.png")
plt.show()
plt.close(fig)

print(f"Graphiques enregistrés dans : {dossier_images}")